<a href="https://colab.research.google.com/github/sanjayraja21/Used_Car_Price_Polynomial_Regression/blob/main/Used_Car_Price_Polynomial_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# USED CAR PRICE PREDICTION USING POLYNOMIAL RIDGE REGRESSION
# Complete Single-Cell Google Colab Project
# Dataset: Car details .csv
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    PolynomialFeatures
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from google.colab import files


# ------------------------------------------------------------
# 2. UPLOAD DATASET
# ------------------------------------------------------------

print("=" * 70)
print("USED CAR PRICE PREDICTION - POLYNOMIAL REGRESSION")
print("=" * 70)

uploaded = files.upload()

# Automatically find CSV file
csv_files = [file for file in uploaded.keys() if file.lower().endswith(".csv")]

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file was uploaded.")

file_name = csv_files[0]

print("\nDataset loaded:", file_name)

df = pd.read_csv(file_name)

print("\nOriginal Dataset Shape:", df.shape)


# ------------------------------------------------------------
# 3. INITIAL DATASET INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INITIAL DATASET INFORMATION")
print("=" * 70)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTotal Missing Values:", df.isnull().sum().sum())

print("\nDuplicate Rows:", df.duplicated().sum())


# ------------------------------------------------------------
# 4. REMOVE DUPLICATE ROWS
# ------------------------------------------------------------

before_duplicates = len(df)

df = df.drop_duplicates().copy()

after_duplicates = len(df)

print("\nDuplicates Removed:", before_duplicates - after_duplicates)
print("Dataset Shape After Removing Duplicates:", df.shape)


# ------------------------------------------------------------
# 5. CLEAN NUMERICAL COLUMNS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA CLEANING")
print("=" * 70)

# Function to extract the first numerical value from text
def extract_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value)

    match = re.search(r"([\d.]+)", value)

    if match:
        try:
            return float(match.group(1))
        except:
            return np.nan

    return np.nan


# Clean mileage
if "mileage" in df.columns:
    df["mileage"] = df["mileage"].apply(extract_number)

# Clean engine
if "engine" in df.columns:
    df["engine"] = df["engine"].apply(extract_number)

# Clean maximum power
if "max_power" in df.columns:
    df["max_power"] = df["max_power"].apply(extract_number)

# Clean torque
if "torque" in df.columns:
    df["torque_value"] = df["torque"].apply(extract_number)

# Convert seats to numerical
if "seats" in df.columns:
    df["seats"] = pd.to_numeric(df["seats"], errors="coerce")


# ------------------------------------------------------------
# 6. HANDLE INVALID NUMERICAL VALUES
# ------------------------------------------------------------

numeric_clean_columns = [
    "mileage",
    "engine",
    "max_power",
    "torque_value",
    "seats"
]

print("\nMissing values after numerical conversion:")

for col in numeric_clean_columns:
    if col in df.columns:
        print(f"{col}: {df[col].isnull().sum()}")


# ------------------------------------------------------------
# 7. MEAN IMPUTATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MEAN IMPUTATION")
print("=" * 70)

for col in numeric_clean_columns:
    if col in df.columns:

        mean_value = df[col].mean()

        df[col] = df[col].fillna(mean_value)

        print(
            f"{col:<20} Mean used for replacement: "
            f"{mean_value:.2f}"
        )


# ------------------------------------------------------------
# 8. FEATURE ENGINEERING
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)

# Current year used for calculating car age
CURRENT_YEAR = 2026

# Car age
df["car_age"] = CURRENT_YEAR - df["year"]

# Prevent zero age
df["car_age"] = df["car_age"].replace(0, 1)

# Kilometres driven per year
df["km_per_year"] = (
    df["km_driven"] / df["car_age"]
)

# Avoid infinite values
df["km_per_year"] = df["km_per_year"].replace(
    [np.inf, -np.inf],
    np.nan
)

# Power-to-engine relationship
df["power_per_engine"] = (
    df["max_power"] /
    df["engine"].replace(0, np.nan)
)

# Engine size per seat
df["engine_per_seat"] = (
    df["engine"] /
    df["seats"].replace(0, np.nan)
)

# Replace infinite values
df = df.replace([np.inf, -np.inf], np.nan)


# ------------------------------------------------------------
# 9. MEAN FILL ENGINEERED FEATURES
# ------------------------------------------------------------

engineered_numeric = [
    "km_per_year",
    "power_per_engine",
    "engine_per_seat"
]

for col in engineered_numeric:

    if col in df.columns:

        df[col] = df[col].fillna(df[col].mean())


# ------------------------------------------------------------
# 10. EXTRACT CAR BRAND
# ------------------------------------------------------------

# The full "name" column contains many different car models.
# Keeping the complete name can create too many categories.
# Instead, extract the first word as the brand.

df["brand"] = (
    df["name"]
    .astype(str)
    .str.split()
    .str[0]
    .str.lower()
)

print("\nExample extracted brands:")
print(df["brand"].value_counts().head(15))


# ------------------------------------------------------------
# 11. CREATE TORQUE NUMERICAL FEATURE
# ------------------------------------------------------------

# torque_value was already created during cleaning.
# Original torque text will not be used.

# ------------------------------------------------------------
# 12. FINAL FEATURE SELECTION
# ------------------------------------------------------------

numeric_features = [
    "km_driven",
    "mileage",
    "engine",
    "max_power",
    "torque_value",
    "seats",
    "car_age",
    "km_per_year",
    "power_per_engine",
    "engine_per_seat"
]

categorical_features = [
    "fuel",
    "seller_type",
    "transmission",
    "owner",
    "brand"
]

target_column = "selling_price"


# ------------------------------------------------------------
# 13. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = (
    numeric_features +
    categorical_features +
    [target_column]
)

missing_required = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required:

    raise ValueError(
        "Missing required columns: "
        + str(missing_required)
    )


# ------------------------------------------------------------
# 14. REMOVE INVALID TARGET VALUES
# ------------------------------------------------------------

df[target_column] = pd.to_numeric(
    df[target_column],
    errors="coerce"
)

df = df.dropna(
    subset=[target_column]
).copy()

df = df[
    df[target_column] > 0
].copy()


# ------------------------------------------------------------
# 15. CREATE X AND y
# ------------------------------------------------------------

X = df[
    numeric_features +
    categorical_features
]

y = df[target_column]


print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print("\nFinal Dataset Shape:", df.shape)

print("Feature Matrix Shape:", X.shape)

print("Target Shape:", y.shape)


# ------------------------------------------------------------
# 16. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))


# ------------------------------------------------------------
# 17. NUMERICAL PIPELINE
# ------------------------------------------------------------

# IMPORTANT:
# PolynomialFeatures is applied ONLY to numerical variables.
#
# This is better than applying polynomial features after
# one-hot encoding because categorical dummy variables should
# not unnecessarily generate polynomial combinations.

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="mean")
        ),

        (
            "polynomial",
            PolynomialFeatures(
                degree=2,
                include_bias=False
            )
        ),

        (
            "scaler",
            StandardScaler()
        )
    ]
)


# ------------------------------------------------------------
# 18. CATEGORICAL PIPELINE
# ------------------------------------------------------------

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# ------------------------------------------------------------
# 19. COMBINE PREPROCESSING
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numeric_pipeline,
            numeric_features
        ),

        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],

    sparse_threshold=0
)


# ------------------------------------------------------------
# 20. CREATE RIDGE REGRESSION PIPELINE
# ------------------------------------------------------------

model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "ridge",
            Ridge()
        )
    ]
)


# ------------------------------------------------------------
# 21. HYPERPARAMETER TUNING
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HYPERPARAMETER TUNING")
print("=" * 70)

param_grid = {

    # Polynomial degree
    "preprocessor__numerical__polynomial__degree": [
        1,
        2,
        3
    ],

    # Ridge regularization
    "ridge__alpha": [
        0.01,
        0.1,
        1,
        10,
        100
    ]
}


# ------------------------------------------------------------
# 22. GRID SEARCH WITH 5-FOLD CROSS VALIDATION
# ------------------------------------------------------------

grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

print("\nStarting 5-Fold Cross Validation...")

grid_search.fit(
    X_train,
    y_train
)


# ------------------------------------------------------------
# 23. BEST PARAMETERS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST MODEL")
print("=" * 70)

print("\nBest Parameters:")

for parameter, value in grid_search.best_params_.items():
    print(f"{parameter}: {value}")

print(
    "\nBest Cross-Validation R²:",
    round(grid_search.best_score_, 4)
)


# ------------------------------------------------------------
# 24. FINAL MODEL
# ------------------------------------------------------------

best_model = grid_search.best_estimator_


# ------------------------------------------------------------
# 25. MAKE TEST PREDICTIONS
# ------------------------------------------------------------

y_pred = best_model.predict(X_test)


# Prevent negative price predictions
y_pred = np.maximum(y_pred, 0)


# ------------------------------------------------------------
# 26. CALCULATE REGRESSION METRICS
# ------------------------------------------------------------

mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    y_pred
)


# ------------------------------------------------------------
# 27. DISPLAY FINAL RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MODEL PERFORMANCE")
print("=" * 70)

print(f"\nMAE  : ₹{mae:,.2f}")
print(f"RMSE : ₹{rmse:,.2f}")
print(f"R²   : {r2:.4f}")


# ------------------------------------------------------------
# 28. INTERPRET R²
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODEL INTERPRETATION")
print("=" * 70)

if r2 >= 0.80:
    print(
        "\nExcellent result!"
        "\nThe model achieved an R² score above 0.80."
    )

elif r2 >= 0.70:
    print(
        "\nGood result."
        "\nThe model achieved an R² score above 0.70."
    )

else:
    print(
        "\nThe model R² is below 0.70."
        "\nFurther feature engineering may be required."
    )


# ------------------------------------------------------------
# 29. ACTUAL VS PREDICTED DATAFRAME
# ------------------------------------------------------------

results = pd.DataFrame({

    "Actual Price": y_test.values,

    "Predicted Price": y_pred,

    "Absolute Error": np.abs(
        y_test.values - y_pred
    )
})

results["Error Percentage"] = (
    results["Absolute Error"] /
    results["Actual Price"]
) * 100


print("\n" + "=" * 70)
print("ACTUAL VS PREDICTED PRICES")
print("=" * 70)

display(
    results.head(15)
)








# ------------------------------------------------------------
# 32. R² COMPARISON FOR DIFFERENT POLYNOMIAL DEGREES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("POLYNOMIAL DEGREE COMPARISON")
print("=" * 70)

degree_results = []

for degree in [1, 2, 3]:

    degree_pipeline = Pipeline(
        steps=[

            (
                "preprocessor",

                ColumnTransformer(
                    transformers=[

                        (
                            "numerical",

                            Pipeline(
                                steps=[
                                    (
                                        "imputer",
                                        SimpleImputer(
                                            strategy="mean"
                                        )
                                    ),

                                    (
                                        "polynomial",

                                        PolynomialFeatures(
                                            degree=degree,
                                            include_bias=False
                                        )
                                    ),

                                    (
                                        "scaler",
                                        StandardScaler()
                                    )
                                ]
                            ),

                            numeric_features
                        ),

                        (
                            "categorical",

                            categorical_pipeline,

                            categorical_features
                        )
                    ],

                    sparse_threshold=0
                )
            ),

            (
                "ridge",
                Ridge(alpha=10)
            )
        ]
    )

    degree_pipeline.fit(
        X_train,
        y_train
    )

    degree_prediction = degree_pipeline.predict(
        X_test
    )

    degree_prediction = np.maximum(
        degree_prediction,
        0
    )

    degree_r2 = r2_score(
        y_test,
        degree_prediction
    )

    degree_results.append({
        "Polynomial Degree": degree,
        "R² Score": degree_r2
    })


degree_df = pd.DataFrame(
    degree_results
)

degree_df["R² Score"] = degree_df[
    "R² Score"
].round(4)

display(degree_df)





# ------------------------------------------------------------
# 34. SAMPLE CAR PRICE PREDICTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE CAR PRICE PREDICTION")
print("=" * 70)

sample_car = X_test.iloc[[0]]

sample_actual = y_test.iloc[0]

sample_prediction = best_model.predict(
    sample_car
)[0]

sample_prediction = max(
    sample_prediction,
    0
)

print("\nCar Information:")

display(sample_car)

print(
    f"\nActual Selling Price   : ₹{sample_actual:,.2f}"
)

print(
    f"Predicted Selling Price: ₹{sample_prediction:,.2f}"
)

print(
    f"Prediction Difference  : "
    f"₹{abs(sample_actual - sample_prediction):,.2f}"
)


# ------------------------------------------------------------
# 35. SAVE PREDICTIONS
# ------------------------------------------------------------

results.to_csv(
    "car_price_predictions.csv",
    index=False
)

print(
    "\nPrediction results saved as:"
    " car_price_predictions.csv"
)


# ------------------------------------------------------------
# 36. FINAL PROJECT SUMMARY
# ------------------------------------------------------------

best_degree = grid_search.best_params_[
    "preprocessor__numerical__polynomial__degree"
]

best_alpha = grid_search.best_params_[
    "ridge__alpha"
]

best_cv_r2 = grid_search.best_score_

print("\n")
print("=" * 70)
print("PROJECT COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    f"\nDataset Shape          : {df.shape}"
)

print(
    f"Training Samples       : {len(X_train)}"
)

print(
    f"Testing Samples        : {len(X_test)}"
)

print(
    f"Best Polynomial Degree : {best_degree}"
)

print(
    f"Best Ridge Alpha       : {best_alpha}"
)

print(
    f"Cross-Validation R²    : {best_cv_r2:.4f}"
)

print(
    f"Test R²                : {r2:.4f}"
)

print(
    f"MAE                    : ₹{mae:,.2f}"
)

print(
    f"RMSE                   : ₹{rmse:,.2f}"
)

print(
    "Model                  : Polynomial Ridge Regression"
)

print("=" * 70)
print("END OF PROJECT")
print("=" * 70)

USED CAR PRICE PREDICTION - POLYNOMIAL REGRESSION


Saving Car details .csv to Car details  (3).csv

Dataset loaded: Car details  (3).csv

Original Dataset Shape: (8128, 13)

INITIAL DATASET INFORMATION

Columns:
['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner', 'mileage', 'engine', 'max_power', 'torque', 'seats']

First 5 rows:


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           8128 non-null   object 
 1   year           8128 non-null   int64  
 2   selling_price  8128 non-null   int64  
 3   km_driven      8128 non-null   int64  
 4   fuel           8128 non-null   object 
 5   seller_type    8128 non-null   object 
 6   transmission   8128 non-null   object 
 7   owner          8128 non-null   object 
 8   mileage        7907 non-null   object 
 9   engine         7907 non-null   object 
 10  max_power      7913 non-null   object 
 11  torque         7906 non-null   object 
 12  seats          7907 non-null   float64
dtypes: float64(1), int64(3), object(9)
memory usage: 825.6+ KB
None

Missing Values:
name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0

,Actual Price,Predicted Price,Absolute Error,Error Percentage
0,550000,5.647719e+05,14771.937223,2.685807
1,1225000,1.099689e+06,125310.641154,10.229440
2,850000,7.390879e+05,110912.147726,13.048488
3,80000,2.056492e+05,125649.226251,157.061533
4,825000,8.798408e+05,54840.838274,6.647374
5,800000,9.152821e+05,115282.122854,14.410265
6,229999,2.868306e+05,56831.609422,24.709503
7,93150,1.732290e+05,80079.034860,85.967831
8,300000,2.080896e+05,91910.409758,30.636803
9,493000,3.802165e+05,112783.506873,22.876979



POLYNOMIAL DEGREE COMPARISON


,Polynomial Degree,R² Score
0,1,0.7264
1,2,0.8687
2,3,0.8860



SAMPLE CAR PRICE PREDICTION

Car Information:


,km_driven,mileage,engine,max_power,torque_value,seats,car_age,km_per_year,power_per_engine,engine_per_seat,fuel,seller_type,transmission,owner,brand
8077,250000,12.8,2494.0,102.0,20.4,7.0,17,14705.882353,0.040898,356.285714,Diesel,Individual,Manual,First Owner,toyota



Actual Selling Price   : ₹550,000.00
Predicted Selling Price: ₹564,771.94
Prediction Difference  : ₹14,771.94

Prediction results saved as: car_price_predictions.csv


PROJECT COMPLETED SUCCESSFULLY

Dataset Shape          : (6926, 19)
Training Samples       : 5540
Testing Samples        : 1386
Best Polynomial Degree : 2
Best Ridge Alpha       : 10
Cross-Validation R²    : 0.8506
Test R²                : 0.8687
MAE                    : ₹98,478.78
RMSE                   : ₹169,684.63
Model                  : Polynomial Ridge Regression
END OF PROJECT
